# Stages 5 and 7 - Features and baselines

**Stage 5 (features)** runs after the split. TF-IDF is fitted on the training texts only; sentence embeddings come from a fixed pre-trained model; both are built with and without the context turn (ablation A1). Settings: `configs/features.yaml`. **Runtime → GPU** for the embeddings on the full data.

**Stage 7 (baselines B0–B2)** is added below once built.

In [ ]:
# --- Colab setup (skip when running locally) ---
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "https://github.com/WarunaS/knowledge-retention-ml.git"
    !git clone -q {REPO} /content/knowledge-retention-ml || git -C /content/knowledge-retention-ml pull -q
    %pip install -q -e "/content/knowledge-retention-ml[ml]"
    os.environ["KRS_DATA_ROOT"] = "/content/drive/MyDrive/krs/data"
    os.environ["KRS_MODELS_DIR"] = "/content/drive/MyDrive/krs/models"
    os.environ["KRS_RESULTS_DIR"] = "/content/drive/MyDrive/krs/results"

In [ ]:
from krs.features import build_features

report = build_features("features")   # writes data/features/

In [ ]:
import pandas as pd

pd.DataFrame(report["splits"], index=["segments"])

In [ ]:
# Inputs longer than the fine-tuned model's 512-token limit will be cut in Stage 8 (context first)
pd.DataFrame(report["token_lengths"]).T

## Stage 7 - Baselines B0, B1, B2

One yes/no classifier per label. C and the per-label thresholds are chosen on **validation** only (Stage 10). Test scores come in Stage 9. CPU is enough.

In [ ]:
import pandas as pd
from krs.models.runner import run_experiment

results = {rid: run_experiment(rid) for rid in ("B0", "B1", "B2")}
pd.DataFrame(results).T[["macro_f1_knowledge_types", "macro_f1_domains", "macro_f1_mean", "chosen_C"]]